# seattle waterfront tracer test

a 3 m uniform-grid geoclaw run over corrected terrain, with a small buffer around pike place and massless tracers blocked by building footprints. buggy still :(

In [ ]:
%matplotlib inline
from pylab import *
from importlib import reload
from clawpack.visclaw import animation_tools
from IPython.display import HTML
from clawpack.geoclaw import fgout_tools
from scipy.interpolate import RegularGridInterpolator
import pandas as pd
import shapely
from shapely import wkt
import sys
sys.path.insert(0, '../../src')
from geoclaw_debris import debris_tracking
reload(debris_tracking)

# load original geographic building footprints
_df = pd.read_csv('../../SeattleBuildings/Seattle_Inventory.csv')
_df['geometry'] = _df['geometry'].apply(wkt.loads)
building_polys_all = list(_df['geometry'])

In [ ]:
from netCDF4 import Dataset
from prepare_topo import file_hash
with Dataset('pike_place_buffered_fgout.nc') as ds:
    if getattr(ds, 'terrain_sha256', None) != file_hash('seattle_buildings_mhw.tt3'): raise RuntimeError('Flow is stale: run python run_corrected.py to regenerate it on corrected terrain.')
    B_xy = array(ds['B'][0]) 
x, y, t, qoi_arrays = fgout_tools.read_netcdf_arrays('pike_place_buffered_fgout.nc', qois=['h', 'u', 'v'], verbose=True)
print(f'dx={x[1]-x[0]:.6f} deg, dy={y[1]-y[0]:.6f} deg, {len(t)} frames')
X, Y = meshgrid(x, y, indexing='ij')
depth = qoi_arrays['h'].transpose((1, 2, 0))
u_ms  = qoi_arrays['u'].transpose((1, 2, 0))
v_ms  = qoi_arrays['v'].transpose((1, 2, 0))

In [ ]:
lat_mid = 0.5 * (y[0] + y[-1])
m_per_deg_lat = 111000.
m_per_deg_lon = 111000. * cos(radians(lat_mid))
aspect = 1 / cos(radians(lat_mid))

u_degs = u_ms / m_per_deg_lon
v_degs = v_ms / m_per_deg_lat

def make_fcn(qoi):
    interp = RegularGridInterpolator((x, y, t), qoi, method='linear', fill_value=nan, bounds_error=False)
    return lambda xi, yi, ti: interp((xi, yi, ti))

depth_fcn = make_fcn(depth)
u_fcn = make_fcn(u_degs)
v_fcn = make_fcn(v_degs)

def depth_positive_fcn(x, y, t):
    h = depth_fcn(x, y, t)
    return ma.masked_where(h < 0.001, h)

from shapely.geometry import box
from shapely.ops import unary_union
flow_extent = box(x.min(), y.min(), x.max(), y.max())
building_polys = [p for p in building_polys_all if p.intersects(flow_extent)]
building_union = unary_union(building_polys)
print(f'{len(building_polys)} buildings in the flow domain')

In [ ]:
reload(debris_tracking)

lon_seed = linspace(-122.355, -122.338, 25)
lat_seed = linspace(47.606,   47.617,   25)

avg_deg_per_m = 0.5 * (1/m_per_deg_lon + 1/m_per_deg_lat)
L = 3.0 * avg_deg_per_m

debris_list = []
z0_list     = []
for lon in lon_seed:
    for lat in lat_seed:
        ix = argmin(abs(x - lon))
        iy = argmin(abs(y - lat))
        B_here = B_xy[ix, iy]
        if -5.0 <= B_here <= 10.0:
            db = debris_tracking.DebrisObject()
            db.L = [L, L, L]
            db.phi = [pi/2, pi/2, pi/2]
            db.z = (lon, lat, 0.0)
            db.advect = True
            db.rho = 0.0
            xc0, yc0 = db.get_corners()
            tracer_poly = shapely.Polygon(column_stack((xc0, yc0)))
            if not tracer_poly.intersects(building_union):
                debris_list.append(db)
                z0_list.append(db.z)

print(f'seeded {len(debris_list)} tracers')

In [ ]:
import time

domain = [x.min(), x.max(), y.min(), y.max()]
t0 = 60.
dt = 5.0
nsteps = int((600 - t0) / dt)

debris_path_list = []
t_start = time.time()
for i, (db, z0) in enumerate(zip(debris_list, z0_list)):
    path = debris_tracking.make_massless_tracer_path(db, z0, building_union, domain, t0, dt, nsteps, depth_fcn, u_fcn, v_fcn)
    debris_path_list.append(path)
    if (i+1) % 10 == 0:
        elapsed = time.time() - t_start
        rate = elapsed / (i+1)
        remaining = rate * (len(debris_list) - (i+1))
        print(f'{i+1}/{len(debris_list)} tracers — {elapsed:.0f}s elapsed, ~{remaining:.0f}s remaining')

path_x = array([dp.x_path.mean(axis=1) for dp in debris_path_list])
path_y = array([dp.y_path.mean(axis=1) for dp in debris_path_list])
inside_building = shapely.contains_xy(building_union, path_x, path_y)
assert not inside_building.any(), 'ruh roh tracer entered a building footprint'
savez_compressed('pike_place_tracers.npz', x=path_x, y=path_y, times=debris_path_list[0].times)
print(f'done: {len(debris_path_list)} tracers in {time.time()-t_start:.0f}s; no building crossings')

In [ ]:
from clawpack.visclaw import colormaps
from shapely.geometry import MultiPolygon
cmap_water = colormaps.make_colormap({0:[0,1,1], 1:[0,0,1]})

# downtown waterfront
x1, x2 = -122.358, -122.336
y1, y2 = 47.605, 47.618

view_buildings = [p for p in building_polys if p.bounds[0] < x2 and p.bounds[2] > x1 and p.bounds[1] < y2 and p.bounds[3] > y1]
print(f'{len(view_buildings)} buildings in view')

def draw_poly(ax, poly):
    xp, yp = poly.exterior.xy
    ax.fill(xp, yp, color='orange', alpha=0.8, zorder=3)
    ax.plot(xp, yp, color='darkorange', linewidth=0.5, zorder=3)

# render only the viewport
view_ix = where((x >= x1) & (x <= x2))[0]
view_iy = where((y >= y1) & (y <= y2))[0]
view_index = ix_(view_ix, view_iy)
X_view, Y_view = X[view_index], Y[view_index]
B_view = B_xy[view_index]
path_dist = sqrt(((path_x-path_x[:,[0]])*m_per_deg_lon)**2 + ((path_y-path_y[:,[0]])*m_per_deg_lat)**2)
dist_color_max = max(100., 25.*ceil(percentile(path_dist, 99)/25.))
rcParams['animation.embed_limit'] = 50
figs  = []
times = debris_path_list[0].times
for n in range(0, len(times), 2):
    t_n = times[n]
    fig, ax = subplots(figsize=(11, 7))

    ax.pcolormesh(X_view, Y_view, ma.masked_where(B_view <= 0, B_view), cmap='terrain', vmin=0, vmax=15, alpha=0.9)
    ax.pcolormesh(X_view, Y_view, depth_positive_fcn(X_view, Y_view, t_n), cmap=cmap_water, vmin=0, vmax=6)

    for poly in view_buildings:
        if isinstance(poly, MultiPolygon):
            for part in poly.geoms: draw_poly(ax, part)
        else: draw_poly(ax, poly)

    ax.contour(X_view, Y_view, B_view, levels=[0], colors='black', linewidths=0.8)

    x_now, y_now, dist = path_x[:,n], path_y[:,n], path_dist[:,n]
    ax.scatter(x_now, y_now, c=dist, cmap='plasma', s=8, vmin=0, vmax=dist_color_max, zorder=5)

    ax.set_xlim(x1, x2)
    ax.set_ylim(y1, y2)
    ax.set_aspect(aspect)
    ax.set_title(f't = {t_n:.0f} s')
    ax.ticklabel_format(useOffset=False)
    figs.append(fig)
    close(fig)

anim = animation_tools.animate_images(animation_tools.make_images(figs), figsize=(11, 7))
HTML(anim.to_jshtml())